# tiny-ton GEMM Benchmark: Road to cuBLAS

This notebook benchmarks **tiny-ton's Python DSL kernels** against cuBLAS,
following the same GEMM-first progression as Modular's matmul series.

| Kernel | Grid | What it adds | Compiler requirement |
|--------|------|--------------|----------------------|
| **K0** Naive GEMM | `(M*N,)` | One thread per output element; `pid//N`, `pid%N` for 2D index | `//` and `%` operators in JIT |
| **K1** Row GEMM | `(M,)` | One block per row; compile-time K loop | existing (constexpr loop) |
| **K2** Shmem GEMM | `(M/TM, N/TN)` | 2D grid; A and B tiles in shared memory | `program_id(1)`, runtime `scf.for` |
| **K3** Swizzled GEMM | same | XOR-swizzle shmem addresses → no bank conflicts | address helper in JIT |
| **K4** Vectorized GEMM | same | `LDG.128` — 4 floats per load instruction | vectorized load emit |
| **K5** Async GEMM | same | `cp.async` — overlap compute and memory | async copy op |
| **K6** Tensor Core GEMM | same | `mma.sync.m16n8k16` FP16 — 4× throughput | `tt.dot` → warp-level MMA |

Each kernel exposes what the compiler needs next.


## 1. One-time setup — run in your Jetson terminal

`sudo` needs an interactive terminal, so paste these commands once via `ssh jetson`.

**Step 1 — Install LLVM 18 + build tools:**
```bash
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | sudo tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
echo "deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main" | sudo tee /etc/apt/sources.list.d/llvm-18.list
sudo apt-get update -qq
sudo apt-get install -y libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build
pip install pybind11
```

**Step 2 — Clone and build tiny-ton (~3 min):**
```bash
git clone https://github.com/ganeshnj/tiny-ton.git ~/tiny-ton
cd ~/tiny-ton
cmake -G Ninja -S . -B build \\
  -DCMAKE_BUILD_TYPE=Release \\
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \\
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \\
  -DTTN_ENABLE_CUDA=ON \\
  -DTTN_ENABLE_PYTHON=ON \\
  -DCUDAToolkit_ROOT=/usr/local/cuda \\
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
```

When both steps finish, run the check cell below then continue.

In [1]:
import shutil, os

checks = {
    'mlir-opt-18':    shutil.which('mlir-opt-18') is not None,
    'cmake':          shutil.which('cmake') is not None,
    'ninja':          shutil.which('ninja') is not None,
    'tiny-ton build': os.path.exists(os.path.expanduser('~/tiny-ton/build/bindings')),
}
all_ok = True
for name, ok in checks.items():
    tag = 'OK' if ok else 'MISSING  <- run Step 1/2 above first'
    print(f'  {name:20s}  {tag}')
    all_ok = all_ok and ok

if not all_ok:
    raise SystemExit('Setup incomplete.')
print('\nAll checks passed — continue to cell 2.')


  mlir-opt-18           OK
  cmake                 OK
  ninja                 OK
  tiny-ton build        OK

All checks passed — continue to cell 2.


In [2]:
import sys, os, time
import numpy as np

TINY_TON_ROOT = os.path.expanduser('~/tiny-ton')
sys.path.insert(0, os.path.join(TINY_TON_ROOT, 'build', 'bindings'))
sys.path.insert(0, os.path.join(TINY_TON_ROOT, 'python'))

os.environ['TTN_SM_VERSION'] = 'sm_87'   # Jetson Orin Nano — Ampere

import _tiny_ton_core as core
import tiny_ton as tt

print(f'has_cuda()   = {core.has_cuda()}')
print(f'SM version   = {os.environ["TTN_SM_VERSION"]}')
print(f'NumPy        = {np.__version__}')


has_cuda()   = True
SM version   = sm_87
NumPy        = 1.21.5


In [3]:
# ── Timing helper ────────────────────────────────────────────────────────────
# tiny-ton copies numpy arrays to/from device synchronously on each call,
# so wall-clock time captures the full H2D + kernel + D2H roundtrip.

def tflops(m, n, k, ms):
    """TFLOPS for a M×K @ K×N matmul given elapsed milliseconds."""
    return 2.0 * m * n * k / (ms * 1e-3) / 1e12

def benchmark_ttn(kernel_fn, grid, args, warmup=10, reps=50):
    """
    Warm up the JIT, then time `reps` consecutive calls.
    Returns average milliseconds per call.
    """
    for _ in range(warmup):
        kernel_fn[grid](*args)
    t0 = time.perf_counter()
    for _ in range(reps):
        kernel_fn[grid](*args)
    return (time.perf_counter() - t0) / reps * 1000.0

print('Timing helpers ready.')


Timing helpers ready.


## 3. cuBLAS Reference Numbers

From `cublas_matmul.ipynb` (already run on this Jetson Orin Nano).  
These are the targets every tiny-ton kernel level is measured against.

| Shape | FP32 SGEMM | FP16 HGEMM (tensor cores) |
|-------|-----------|---------------------------|
| 64³   | ~0.005 TF  | ~0.02 TF                  |
| 128³  | ~0.03 TF   | ~0.12 TF                  |
| 256³  | ~0.2 TF    | ~0.7 TF                   |
| 512³  | ~0.6 TF    | ~2.5 TF                   |
| 1024³ | ~1.2 TF    | ~5 TF                     |
| 2048³ | ~1.8 TF    | ~9 TF                     |
| 4096³ | ~2.0 TF    | ~12 TF                    |

**Goal**: match the FP32 SGEMM column first (Phase A+B), then FP16 with tensor cores (Phase C).


## 4. K0 — Naive GEMM

```
Grid:  (M * N,)  — one block per output element
Block: K threads  (block size = K)
```

Each block computes one scalar `C[row, col]`. The flat `pid` is decomposed into
`(row, col)` using `//` and `%`. All K elements are loaded in one shot and reduced:

```python
row = pid // N
col = pid % N
tid = tt.arange(0, K)        # K threads, one per K element
acc = tt.reduce_sum(tt.load(A[row,:]) * tt.load(B[:,col]))
```

No loop, no tiling. K must fit in a single block (≤ 1024 on Ampere).

**Bottleneck**: zero data reuse — every block independently loads its A row and B column.


In [4]:
# ── K0: Naive GEMM ────────────────────────────────────────────────────────────
# C[M,N] = A[M,K] @ B[K,N]
# Grid (M*N,): one block per output element.
# Block size = K: each thread handles one element of the K reduction.
# No loop, no tiling — truly naive.

@tt.jit
def naive_gemm(A_ptr, B_ptr, C_ptr,
               M: tt.constexpr, N: tt.constexpr, K: tt.constexpr):
    pid = tt.program_id(0)
    row = pid // N
    col = pid % N
    tid = tt.arange(0, K)
    a   = tt.load(A_ptr + row * K + tid)
    b   = tt.load(B_ptr + tid * N + col)
    tt.store(C_ptr + row * N + col, tt.reduce_sum(a * b))


# ── Correctness ───────────────────────────────────────────────────────────────
np.random.seed(0)
print('K0 correctness:')
for M, N, K in [(4, 4, 8), (8, 8, 16), (16, 16, 32), (32, 32, 64)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    naive_gemm[(M * N,)](A.copy(), B.copy(), C, M, N, K)
    err = float(np.max(np.abs(C - A @ B)))
    status = 'PASS' if err < 1e-3 else 'FAIL'
    print(f'  M={M:3d} N={N:3d} K={K:3d}  max_err={err:.2e}  {status}')

# ── Benchmark ─────────────────────────────────────────────────────────────────
# K = block size, must be ≤ 1024.  Use square shapes to match cuBLAS reference.
k0_results = {}
print(f'\n{"Shape (MxNxK)":>16s}  {"ms":>8s}  {"TFLOPS":>10s}  {"cuBLAS FP32":>12s}  {"Gap":>6s}')
print('-' * 62)
cublas_ref = {(32,32,32): 0.005, (64,64,64): 0.005, (128,128,128): 0.03, (256,256,256): 0.2, (512,512,512): 0.6}
for M, N, K in [(32,32,32), (64,64,64), (128,128,128), (256,256,256), (512,512,512)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    ms  = benchmark_ttn(naive_gemm, grid=(M * N,), args=(A, B, C, M, N, K), warmup=5, reps=20)
    tf  = tflops(M, N, K, ms)
    k0_results[(M, N, K)] = tf
    cub = cublas_ref.get((M, N, K), float('nan'))
    gap = f'{cub/tf:.0f}x' if tf > 0 and not np.isnan(cub) else 'n/a'
    shape = f'{M}x{N}x{K}'
    print(f'{shape:>16s}  {ms:8.4f}  {tf:10.6f}  {cub:12.4f}  {gap:>6s}')


K0 correctness:
  M=  4 N=  4 K=  8  max_err=9.54e-07  PASS
  M=  8 N=  8 K= 16  max_err=1.91e-06  PASS
  M= 16 N= 16 K= 32  max_err=2.86e-06  PASS
  M= 32 N= 32 K= 64  max_err=5.72e-06  PASS

   Shape (MxNxK)        ms      TFLOPS   cuBLAS FP32     Gap
--------------------------------------------------------------
        32x32x32  102.1302    0.000001        0.0050   7792x
        64x64x64  103.2347    0.000005        0.0050    985x
     128x128x128  104.5282    0.000040        0.0300    748x
     256x256x256  116.1996    0.000289        0.2000    693x
     512x512x512  196.6529    0.001365        0.6000    440x


**What's limiting K0:**
- Zero data reuse: block at `(row, col)` reads `A[row,:]` and `B[:,col]` independently.
  Adjacent blocks in the same row re-read the same `B` column.
- H2D + D2H per call dominates the measured time for small matrices.

**What K1 will add:** one block per output *row* computes all N columns in one pass —
N outputs per block instead of 1, so the A row load is amortised over N dot-products.


## 5. K1 — Row GEMM (one block per row)

```
Grid:  (M,)      — one block per output row
Block: TILE_K threads
```

One block computes an entire row of C. For each column it accumulates a dot product
over K tiles using the same A row load — the A row is read once and reused for all N
output values. N× better A reuse compared to K0.

Both the column loop (`for col in range(N)`) and K tile loop (`for k0 in range(0,K,TILE_K)`)
are constexpr-unrolled by the JIT — all bounds must be `tt.constexpr`.

**Bottleneck:** N must be a compile-time constant, so IR grows linearly with N.
No column parallelism between blocks. B columns re-read independently per block.


In [ ]:
# ── K1: Row GEMM ──────────────────────────────────────────────────────────────
# C[M,N] = A[M,K] @ B[K,N]
# Grid (M,): one block per row.
# The block loads A[row,:] once and computes all N output columns.
# Both loops are constexpr-unrolled — N and K must be tt.constexpr.

@tt.jit
def row_gemm(A_ptr, B_ptr, C_ptr,
             K: tt.constexpr, N: tt.constexpr, TILE_K: tt.constexpr):
    row = tt.program_id(0)
    tid = tt.arange(0, TILE_K)
    for col in range(N):
        acc = 0.0
        for k0 in range(0, K, TILE_K):
            a_tile = tt.load(A_ptr + row * K + k0 + tid)
            b_tile = tt.load(B_ptr + k0 * N + col + tid * N)
            acc    = acc + tt.reduce_sum(a_tile * b_tile)
        tt.store(C_ptr + row * N + col, acc)


# ── Correctness ───────────────────────────────────────────────────────────────
np.random.seed(3)
print('K1 correctness:')
for M, N, K, TK in [(4, 4, 8, 4), (8, 8, 16, 8), (16, 16, 32, 16), (32, 32, 64, 16)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    row_gemm[(M,)](A.copy(), B.copy(), C, K, N, TK)
    err = float(np.max(np.abs(C - A @ B)))
    status = 'PASS' if err < 1e-3 else 'FAIL'
    print(f'  M={M:3d} N={N:3d} K={K:3d} TILE_K={TK:2d}  max_err={err:.2e}  {status}')

# ── Benchmark ─────────────────────────────────────────────────────────────────
# Keep N small — IR grows linearly with N (constexpr unrolling).
k1_results = {}
print(f'\n{"Shape (MxNxK)":>16s}  {"ms":>8s}  {"TFLOPS":>10s}  {"vs K0":>6s}  {"cuBLAS FP32":>12s}  {"Gap":>6s}')
print('-' * 72)
cublas_ref = {(32,32,32): 0.005, (64,64,64): 0.005, (128,128,128): 0.03, (256,256,256): 0.2}
for M, N, K, TK in [(32,32,32,16), (64,64,64,16), (128,128,128,16), (256,256,256,16)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    ms  = benchmark_ttn(row_gemm, grid=(M,), args=(A, B, C, K, N, TK), warmup=5, reps=20)
    tf  = tflops(M, N, K, ms)
    k1_results[(M, N, K)] = tf
    cub  = cublas_ref.get((M, N, K), float('nan'))
    gap  = f'{cub/tf:.0f}x' if tf > 0 and not np.isnan(cub) else 'n/a'
    tf_k0 = k0_results.get((M, N, K), None)
    vs_k0 = f'{tf/tf_k0:.1f}x' if tf_k0 and tf_k0 > 0 else 'n/a'
    shape = f'{M}x{N}x{K}'
    print(f'{shape:>16s}  {ms:8.4f}  {tf:10.6f}  {vs_k0:>6s}  {cub:12.4f}  {gap:>6s}')


K1 correctness:
  M=  4 N=  4 K=  8 TILE_K= 4  max_err=4.77e-07  PASS
  M=  8 N=  8 K= 16 TILE_K= 8  max_err=9.54e-07  PASS
  M= 16 N= 16 K= 32 TILE_K=16  max_err=3.81e-06  PASS
  M= 32 N= 32 K= 64 TILE_K=16  max_err=5.72e-06  PASS

   Shape (MxNxK)        ms      TFLOPS   vs K0   cuBLAS FP32     Gap
------------------------------------------------------------------------
        32x32x32  103.0085    0.000001    1.0x        0.0050   7859x
        64x64x64  104.2920    0.000005    1.0x        0.0050    995x
     128x128x128  108.9879    0.000038    1.0x        0.0300    780x


**What's limiting K1:**
- N must be a compile-time constant — IR grows linearly with N×(K/TILE_K) unrolled loop bodies.
- All N columns computed sequentially in one block — no inter-block column parallelism.
- B columns re-read from global memory independently per block.

**What K2 needs (compiler changes):**
- `tt.program_id(1)` — 2D grid so blocks tile both the M and N dimensions.
- `scf.for` runtime K loop — so K can be a runtime arg, eliminating IR explosion.
- Shared memory for the B tile — one load per block, shared across all rows in the tile.


## 6. Gap Analysis

K0 and K1 vs cuBLAS at the shapes measured above.


In [ ]:
# ── Gap analysis — K0 and K1 vs cuBLAS ────────────────────────────────────────
cublas_fp32 = {64: 0.005, 128: 0.03, 256: 0.2, 512: 0.6, 1024: 1.2, 2048: 1.8, 4096: 2.0}
cublas_fp16_peak = 12.0

print('=' * 68)
print(' tiny-ton vs cuBLAS — Jetson Orin Nano (Ampere sm_87, FP32)')
print('=' * 68)
print(f'{"Kernel":16s}  {"Shape":12s}  {"tiny-ton TF":>12s}  {"cuBLAS TF":>10s}  {"Gap":>6s}')
print('-' * 68)

for label, results in [('K0  naive GEMM', k0_results), ('K1  row GEMM', k1_results)]:
    if not results:
        continue
    best = max(results, key=results.get)
    tf   = results[best]
    sz   = best[0]
    cub  = cublas_fp32.get(sz, float('nan'))
    gap  = f'{cub/tf:.0f}x' if tf > 0 else 'n/a'
    cub_s = f'{cub:.4f}' if not np.isnan(cub) else '  —  '
    print(f'{label:16s}  {f"{sz}³":12s}  {tf:12.6f}  {cub_s:>10s}  {gap}')

print()
best_tf = max(k1_results.values()) if k1_results else (max(k0_results.values()) if k0_results else 0)
if best_tf > 0:
    print(f'Gap to cuBLAS FP32 peak (4096³):  {cublas_fp32[4096]/best_tf:,.0f}x')
    print(f'Gap to cuBLAS FP16 peak (4096³):  {cublas_fp16_peak/best_tf:,.0f}x')

print()
print('Main gap sources:')
print('  1. H2D + D2H transfer dominates at small sizes')
print('  2. No data reuse across blocks — each block re-reads B independently')
print('  3. No 2D output tiling — requires program_id(1) + scf.for (K2)')


 tiny-ton vs cuBLAS — Jetson Orin Nano (Ampere sm_87, FP32)
Kernel            Shape          tiny-ton TF   cuBLAS TF     Gap
--------------------------------------------------------------------
K0  naive GEMM    512³              0.001336      0.6000  449x

Gap to cuBLAS FP32 peak (4096³):  1,497x
Gap to cuBLAS FP16 peak (4096³):  8,984x

Main gap sources:
  1. H2D + D2H transfer dominates at small sizes
  2. No data reuse — each block re-reads A row and B column independently
  3. No 2D output tiling — requires program_id(1) + scf.for (K2)


## 6. Roadmap to cuBLAS Performance

---

### Phase A — Remove size limits (requires compiler work)

| Kernel | What to add to tiny-ton | Estimated impact |
|--------|--------------------------|-----------------|
| K1 | One block per row; compile-time N and K loops | baseline for comparison |
| K2 | `program_id(1)` (2D grid) + `scf.for` (runtime K loop) + shared A+B | 10–50× |
| K3 | XOR-swizzle shmem address → eliminate bank conflicts | 1.5–2× |

### Phase B — Memory bandwidth (Ampere microarch)

| Kernel | What to add | Estimated impact |
|--------|-------------|-----------------|
| K4 | `LDG.128` vectorized loads (4 floats/instr) | 1.5–2× |
| K5 | `cp.async` — overlap DMA with compute | 1.5–3× |

### Phase C — Tensor Cores

| Kernel | What to add | Estimated impact |
|--------|-------------|-----------------|
| K6 | `mma.sync.m16n8k16` (`tt.dot` → warp MMA) + `ldmatrix` | 4–8× FP16 |

---

### Expected progress

```
K0 today         < 0.001 TF  (256³, dominated by H2D overhead)
  → K1 (row GEMM)  similar TF, better data reuse within block
  → K2 (shmem)     0.1–0.5 TF  (large sizes, FP32, compute-bound)
  → K4-K5          1–2 TF  (approaching cuBLAS FP32 target)
  → K6 (TC)        6–12 TF  (cuBLAS FP16 with Tensor Cores)

cuBLAS FP32 peak   ~2 TF  (Jetson, 4096³)
cuBLAS FP16 peak  ~12 TF  (tensor cores)
```


<!-- placeholder -->


In [ ]:
print('Done.')

Done.


<!-- placeholder -->

In [ ]:
pass


<!-- placeholder -->


In [ ]:
print('Notebook complete.')


Notebook complete.
